In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# GCN + DEVIGN (TOKEN GRAPH BASELINE)
# Dataset: dcn-dataset
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DATASET PATH
# ============================================================
DATASET_PATH = "/kaggle/input/dcn-dataset"
print("Files:", os.listdir(DATASET_PATH))

# ============================================================
# LOAD DEVIGN
# ============================================================
def load_devign_csv(path):
    df = pd.read_csv(path)
    if "func" in df.columns:
        df["code"] = df["func"]
    if "target" in df.columns:
        df["label"] = df["target"]
    df = df[["code", "label"]]
    df["label"] = df["label"].astype(int)
    return df

train_df = load_devign_csv(f"{DATASET_PATH}/devignx_train.csv")
test_df  = load_devign_csv(f"{DATASET_PATH}/devignx_test.csv")

print("\nLabel distribution:")
print(train_df["label"].value_counts())

# ============================================================
# CLASS WEIGHTS (CRITICAL)
# ============================================================
counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [counts[1] / counts.sum(), counts[0] / counts.sum()],
    dtype=torch.float
).to(device)

# ============================================================
# TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
MAX_LEN = 200
WINDOW = 4  # co-occurrence window

# ============================================================
# GRAPH DATASET
# ============================================================
class DevignGraphDataset(Dataset):
    def __init__(self, df):
        self.codes = df["code"].tolist()
        self.labels = df["label"].tolist()

    def build_adj(self, tokens):
        L = len(tokens)
        adj = torch.zeros((L, L))
        for i in range(L):
            for j in range(max(0, i-WINDOW), min(L, i+WINDOW+1)):
                adj[i, j] = 1
        return adj

    def __getitem__(self, idx):
        tokens = tokenizer.encode(
            self.codes[idx],
            truncation=True,
            max_length=MAX_LEN,
            add_special_tokens=False
        )

        x = torch.tensor(tokens)
        adj = self.build_adj(tokens)
        label = torch.tensor(self.labels[idx])

        return x, adj, label

    def __len__(self):
        return len(self.codes)

def collate_fn(batch):
    xs, adjs, labels = zip(*batch)
    max_len = max(len(x) for x in xs)

    X = torch.zeros(len(xs), max_len, dtype=torch.long)
    A = torch.zeros(len(xs), max_len, max_len)

    for i in range(len(xs)):
        X[i, :len(xs[i])] = xs[i]
        A[i, :adjs[i].shape[0], :adjs[i].shape[1]] = adjs[i]

    return X.to(device), A.to(device), torch.tensor(labels).to(device)

train_loader = DataLoader(
    DevignGraphDataset(train_df),
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    DevignGraphDataset(test_df),
    batch_size=16,
    collate_fn=collate_fn
)

# ============================================================
# GCN MODEL
# ============================================================
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        deg = adj.sum(dim=-1, keepdim=True) + 1e-6
        x = torch.bmm(adj, x) / deg
        return self.fc(x)

class GCN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gcn1 = GCNLayer(embed_dim, 128)
        self.gcn2 = GCNLayer(128, 128)
        self.fc = nn.Linear(128, 2)

    def forward(self, x, adj):
        x = self.embed(x)
        x = F.relu(self.gcn1(x, adj))
        x = F.relu(self.gcn2(x, adj))
        x = x.mean(dim=1)  # graph pooling
        return self.fc(x)

model = GCN(tokenizer.vocab_size).to(device)

# ============================================================
# TRAINING SETUP
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
EPOCHS = 8

# ============================================================
# TRAIN
# ============================================================
print("\nTraining GCN...")
for epoch in range(EPOCHS):
    model.train()
    loss_sum = 0
    for X, A, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X, A), y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss_sum/len(train_loader):.4f}")

# ============================================================
# EVALUATION
# ============================================================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X, A, y in test_loader:
        preds = torch.argmax(model(X, A), dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nPrediction distribution:", np.unique(y_pred, return_counts=True))

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("\n===== GCN DEVIGN RESULTS =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred, zero_division=0))
print("FPR      :", fp / (fp + tn) if (fp + tn) > 0 else 0)
print("Confusion Matrix:", tn, fp, fn, tp)


Device: cuda
GPU: Tesla T4
Files: ['Devignx_validation.csv', 'devignx_test.csv', 'devignx_train.csv']

Label distribution:
label
0    10356
1     8766
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


Training GCN...
Epoch 1/8 | Loss: 0.6908
Epoch 2/8 | Loss: 0.6862
Epoch 3/8 | Loss: 0.6835
Epoch 4/8 | Loss: 0.6809
Epoch 5/8 | Loss: 0.6778
Epoch 6/8 | Loss: 0.6748
Epoch 7/8 | Loss: 0.6721
Epoch 8/8 | Loss: 0.6680

Prediction distribution: (array([0, 1]), array([1291, 1441]))

===== GCN DEVIGN RESULTS =====
Accuracy : 0.5699121522693997
Precision: 0.5267175572519084
Recall   : 0.6062300319488818
F1 Score : 0.5636836242109172
FPR      : 0.4608108108108108
Confusion Matrix: 798 682 493 759
